<a href="https://colab.research.google.com/github/Ghalaahmed/Nukbah/blob/main/LogisticRegression/logistic_regression_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Logistic Regression — Riyadh Restaurants ML Project

## Project Overview

This notebook applies a Logistic Regression model to classify Riyadh restaurants into two classes:

- Highly Rated Restaurant (1)
- Lower Rated Restaurant (0)

## Target Variable

The target variable is `Restaurant_Class`.

The threshold value was determined using the median rating of the dataset to create balanced classes for the classification task.

Restaurants with ratings greater than or equal to the median rating were classified as Highly Rated Restaurants (1), while restaurants with ratings below the median rating were classified as Lower Rated Restaurants (0).

The `rating` column is used only to create the target variable and is not included as an input feature during model training to avoid data leakage.


## Step 1 — Import Required Libraries

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix, classification_report

## Step 2 — Load Dataset

Upload the dataset file to Google Colab before running this notebook.

Recommended file name:

`riyadh_restaurants.csv`


In [2]:
df = pd.read_csv("riyadh_restaurants.csv")

df.head()

,price,likes,tips,photos,ratingSignals,rating,Restaurant_Class
0,1,29.0,1,90,32.0,8.9,1
1,1,22.0,2,13,24.0,8.6,1
2,1,24.0,1,27,33.0,7.3,1
3,1,0.0,0,6,0.0,8.3,1
4,1,1.0,0,16,1.0,8.1,1


## Step 3 — Select Required Columns

The selected columns are used for data cleaning, feature selection, and target variable creation.

The `rating` column is included at this stage only to create the target variable. It will be removed from the input features before model training.


In [3]:
selected_columns = [
    "price",
    "likes",
    "tips",
    "photos",
    "ratingSignals",
    "rating"
]

df = df[selected_columns]

df.head()

,price,likes,tips,photos,ratingSignals,rating
0,1,29.0,1,90,32.0,8.9
1,1,22.0,2,13,24.0,8.6
2,1,24.0,1,27,33.0,7.3
3,1,0.0,0,6,0.0,8.3
4,1,1.0,0,16,1.0,8.1


## Step 4 — Data Cleaning and Preprocessing

The data cleaning process includes:

- Removing duplicated records
- Encoding the price level into numerical values
- Converting selected attributes into numeric format
- Removing rows with missing or invalid values


In [4]:
df = df.drop_duplicates()

price_mapping = {
    "Cheap": 1,
    "Moderate": 2,
    "Expensive": 3,
    "Very Expensive": 4,
    "$": 1,
    "$$": 2,
    "$$$": 3,
    "$$$$": 4
}

df["price"] = df["price"].replace(price_mapping)

numeric_columns = [
    "price",
    "likes",
    "tips",
    "photos",
    "ratingSignals",
    "rating"
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.dropna()

print("Rows after cleaning:", len(df))
df.info()

Rows after cleaning: 7294
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7294 entries, 0 to 7293
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   price          7294 non-null   int64  
 1   likes          7294 non-null   float64
 2   tips           7294 non-null   int64  
 3   photos         7294 non-null   int64  
 4   ratingSignals  7294 non-null   float64
 5   rating         7294 non-null   float64
dtypes: float64(3), int64(3)
memory usage: 342.0 KB


## Step 5 — Create Target Variable

A new target variable called `Restaurant_Class` is created using the median rating of the dataset.

Class labels:

- 1 = Highly Rated Restaurant
- 0 = Lower Rated Restaurant

Using the median rating helps create balanced classes for the classification task.


In [5]:
threshold = df["rating"].median()

df["Restaurant_Class"] = df["rating"].apply(
    lambda x: 1 if x >= threshold else 0
)

print("Selected threshold:", threshold)
print(df["Restaurant_Class"].value_counts())

df[["rating", "Restaurant_Class"]].head()

Selected threshold: 7.7
Restaurant_Class
1    3677
0    3617
Name: count, dtype: int64


,rating,Restaurant_Class
0,8.9,1
1,8.6,1
2,7.3,0
3,8.3,1
4,8.1,1


## Step 6 — Define Features and Target

The input features are selected after data cleaning.

The `rating` column is excluded from the input features because it was used only to create the target variable. This avoids data leakage.


In [6]:
X = df[[
    "price",
    "likes",
    "tips",
    "photos",
    "ratingSignals"
]]

y = df["Restaurant_Class"]

## Step 7 — Split Dataset

The dataset is split into:

- 80% training data
- 20% testing data

Stratified splitting is used to preserve the class distribution in both training and testing sets.


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set size:", len(X_train))
print("Testing set size:", len(X_test))
print("\nClass distribution:")
print(y.value_counts())

Training set size: 5835
Testing set size: 1459

Class distribution:
Restaurant_Class
1    3677
0    3617
Name: count, dtype: int64


## Step 8 — Build and Train Logistic Regression Model

The Logistic Regression model is trained using the training dataset.


In [8]:
logistic_model = LogisticRegression(max_iter=1000)

logistic_model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

## Step 9 — Generate Predictions

The trained model is used to predict restaurant classes on the testing dataset.


In [9]:
y_pred = logistic_model.predict(X_test)

## Step 10 — Model Evaluation

The model is evaluated using:

- Accuracy
- Precision
- Recall
- Confusion Matrix
- Classification Report


In [10]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("Confusion Matrix:")
print(conf_matrix)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Lower Rated Restaurant", "Highly Rated Restaurant"]
    )
)

Accuracy: 0.8574366004112406
Precision: 0.916403785488959
Recall: 0.7894021739130435
Confusion Matrix:
[[670  53]
 [155 581]]

Classification Report:
                         precision    recall  f1-score   support

 Lower Rated Restaurant       0.81      0.93      0.87       723
Highly Rated Restaurant       0.92      0.79      0.85       736

               accuracy                           0.86      1459
              macro avg       0.86      0.86      0.86      1459
           weighted avg       0.86      0.86      0.86      1459



## Step 11 — Save Cleaned Dataset

The cleaned dataset is saved for documentation and future use.


In [11]:
df.to_csv("riyadh_restaurants_cleaned_median_threshold.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
